# Multi-label 불량 — 약지도 patch 검출 (per-type memory)

**문제 정의(확정):** 이미지 단위 **multi-label**(불량 타입 집합), **위치 라벨 없음**, 불량은 **공간 분리**,
그리고 **others**(미지 불량). frozen DINO patch feature. 단일 라벨 classification은 폐기.

**접근(약지도, 학습형 MIL 없이):**
```
① anomaly (정상 대비)               → 이상 patch/region 후보      [무감독]
② 타입별 memory (단일라벨 이미지의 이상 patch로 seed)
     each 이상 patch → 타입별 cosine 매칭 + 임계
        가까운 타입 present (여러 개 동시 = multi-label)
        어떤 타입과도 멀다 → others
③ 이미지 라벨 = patch 타입들의 집합
```
- 공간 분리 → 이상 patch는 사실상 **단일 타입** → 전역 pooling이 못 가르는 co-occurrence를 자동 분리.
- **shortcut 검증:** 이미지-전역 baseline vs patch-raw vs patch-**residual**(정상참조 차감=맥락제거) 의
  **타입별 AP** 비교. residual > raw ≫ 전역 이면 "전역은 맥락 shortcut" 확정.

> feature 는 service `?include=patch`. **IMG_SIZE** 로 입력 해상도 조절(1k+ → 미세 결함 유리, patch=IMG_SIZE/16).



In [ ]:
import os, sys, hashlib
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "inspection").is_dir():
    REPO = REPO.parent
for p in (str(REPO), str(REPO / "dino_v3")):
    if p not in sys.path:
        sys.path.insert(0, p)
from inspection.service.client_example import get_patch_features, _list_images

# ============================ 설정 ============================
SERVER    = os.environ.get("EM_SERVER", "http://localhost:8000")
DATA_ROOT = os.environ.get("EM_DATA_ROOT", "/data/classes")
IMG_SIZE  = int(os.environ.get("EM_IMG_SIZE", "0")) or None   # 예: 1024 (patch 64x64). None=서버기본
NORMAL    = "normal"
SEP       = "+"      # multi-label 폴더명 구분자. 예: class1+class3/ → {class1,class3}
SEED      = 0
SPLIT     = 0.5      # support 비율
K_NN      = 3
CORESET   = 4000     # 정상 메모리
REGION_PCT = 99.0    # 이상 patch 임계 = 정상 patch anomaly 분포 퍼센타일
MEM_SIZE  = 3000     # 타입별 memory coreset 상한
TOPK      = 5        # 타입 점수 = 이상 patch 중 상위 TOPK 매칭 평균
LOCALIZER = os.environ.get("EM_LOCALIZER", "knn")   # "knn"(무학습) | "recon"(transformer, torch)
# recon(Dinomaly-lite) 하이퍼: bottleneck, 층수, epoch, noise, hard-mining 유지비율(1.0=off, 낮출수록 어려운 토큰집중)
RECON_M, RECON_LAYERS, RECON_EPOCHS, RECON_NOISE, RECON_HARDQ = 256, 2, 60, 0.3, 1.0
CACHE     = Path(os.environ.get("EM_CACHE", "./_patchcache"))

print("SERVER=", SERVER, "| IMG_SIZE=", IMG_SIZE or "(서버기본)", "| LOCALIZER=", LOCALIZER, "| DATA_ROOT=", DATA_ROOT)


## 1. 라벨 로드 (폴더명 = 타입 집합)

- `normal/` → 정상(빈 집합).  `class1/` → {class1}(단일라벨).  `class1+class3/` → {class1,class3}(멀티라벨).
- **타입별 memory 는 단일라벨 이미지로만 seed** (깨끗한 양성). 멀티라벨은 평가·임계보정에 사용.
- (CSV manifest 를 쓰려면 `load_dataset` 만 교체하면 됨: path→타입집합.)


In [ ]:
def load_dataset(root, sep=SEP, normal=NORMAL):
    items = []  # (path, frozenset(types))
    for d in sorted(Path(root).iterdir()):
        if not d.is_dir():
            continue
        labels = frozenset() if d.name == normal else frozenset(d.name.split(sep))
        for p in _list_images(str(d)):
            items.append((p, labels))
    return items

def split_items(items, split, seed):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(items))
    k = int(len(items) * split)
    S = [items[i] for i in idx[:k]]; Q = [items[i] for i in idx[k:]]
    return S, Q

items = load_dataset(DATA_ROOT)
types = sorted({t for _, ls in items for t in ls})
sup_items, qry_items = split_items(items, SPLIT, SEED)

def _dist(its):
    from collections import Counter
    c = Counter()
    for _, ls in its:
        c["normal" if not ls else ("multi" if len(ls) > 1 else next(iter(ls)))] += 1
    return dict(c)

print("타입:", types)
print("support 분포:", _dist(sup_items))
print("query   분포:", _dist(qry_items))
sl = {t: sum(1 for _, ls in sup_items if ls == frozenset({t})) for t in types}
print("타입별 support 단일라벨 이미지 수(memory seed 재료):", sl)


In [ ]:
def _key(p):
    return hashlib.sha1((str(Path(p).resolve()) + f"@{IMG_SIZE}").encode()).hexdigest()[:16]

def fetch_patch(paths, batch=8):
    CACHE.mkdir(parents=True, exist_ok=True)
    todo = [p for p in paths if not (CACHE / f"{_key(p)}.npz").exists()]
    for i in range(0, len(todo), batch):
        got = get_patch_features(SERVER, todo[i:i + batch], batch=batch, img_size=IMG_SIZE)
        for p, (pf, h, w, _n) in zip(todo[i:i + batch], got):
            np.savez(CACHE / f"{_key(p)}.npz", pf=pf.astype(np.float32), hw=np.array([h, w]))
        print(f"  fetched {min(i+batch,len(todo))}/{len(todo)}", end="\r")
    out = {}
    for p in paths:
        d = np.load(CACHE / f"{_key(p)}.npz"); out[p] = (d["pf"], int(d["hw"][0]), int(d["hw"][1]))
    return out

feat = fetch_patch([p for p, _ in items])
print("\npatch feature 취득 완료:", len(feat), "장  | grid(예)=", feat[items[0][0]][1], "x", feat[items[0][0]][2])


## 2. 정상 localizer(kNN ↔ recon 토글) + 이상 patch + 타입 feature

- **anomaly localizer** (`LOCALIZER`):
  - `knn`  : 정상 coreset 메모리와의 kNN → `1 − sim` (무학습).
  - `recon`: frozen DINO feature를 정상만으로 재구성하는 **transformer 디코더**(spatial attention +
    noisy bottleneck = Dinomaly-lite) → `1 − cosine(재구성, 원본)`. `RECON_HARDQ<1`이면 hard-mining.
    (torch 필요. 앞 실험에서 **recon만 위치를 정확히** 잡았음 → 여기에 per-type residual typing 결합이 유망 조합.)
- **이상 patch**: anomaly > τ_region(정상 분포 퍼센타일).
- **타입 feature**:
  - `raw`   : 이상 patch 원본 feature (맥락 포함 → shortcut 취약).
  - `residual`: 원본 − (최근접 정상 메모리 평균) = **맥락 제거된 이상 방향** (localizer와 무관, 항상 kNN 참조).


In [ ]:
def l2(x):
    x = np.asarray(x, np.float32)
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def coreset(feats, size, cap=20000, seed=0):
    feats = l2(feats); n = len(feats)
    if n <= size:
        return feats
    r = np.random.default_rng(seed)
    if n > cap:
        feats = feats[r.choice(n, cap, replace=False)]; n = len(feats)
    sel = [int(r.integers(n))]; mx = feats @ feats[sel[0]]
    for _ in range(size - 1):
        i = int(np.argmin(mx)); sel.append(i); mx = np.maximum(mx, feats @ feats[i])
    return feats[np.array(sel)]

# 정상 메모리 (support 정상 patch) — kNN localizer & residual 참조 공용
NORMAL_M = coreset(np.vstack([feat[p][0] for p, ls in sup_items if not ls]), CORESET)

def nearest_normal(P):
    s = l2(P) @ NORMAL_M.T
    idx = np.argpartition(-s, K_NN, axis=1)[:, :K_NN]
    return NORMAL_M[idx].mean(1)

class KNNLocalizer:
    name = "kNN"
    def score(self, pf):
        pf = l2(pf); out = np.empty(len(pf), np.float32)
        for i in range(0, len(pf), 4096):
            s = pf[i:i + 4096] @ NORMAL_M.T; k = min(K_NN, s.shape[1])
            out[i:i + 4096] = 1.0 - np.partition(s, -k, axis=1)[:, -k:].mean(1)
        return out

def build_recon_localizer(normal_imgs):
    import torch
    import torch.nn as nn

    class _Net(nn.Module):
        def __init__(self, D, m, layers, heads=4, drop=0.1):
            super().__init__()
            self.inp = nn.Linear(D, m)
            enc = nn.TransformerEncoderLayer(m, heads, 4 * m, dropout=drop,
                                             batch_first=True, activation="gelu")
            self.dec = nn.TransformerEncoder(enc, layers)
            self.out = nn.Linear(m, D)
        def forward(self, x, noise=0.0):
            z = self.inp(x)
            if noise > 0:
                z = z + torch.randn_like(z) * noise            # noisy bottleneck
            r = self.out(self.dec(z))
            return r / (r.norm(dim=-1, keepdim=True) + 1e-8)

    class ReconLocalizer:
        name = "recon"
        def __init__(self, imgs, m=RECON_M, layers=RECON_LAYERS, epochs=RECON_EPOCHS,
                     noise=RECON_NOISE, hard_q=RECON_HARDQ, lr=1e-3, cap_imgs=600, seed=0):
            D = imgs[0].shape[1]
            self.dev = "cuda" if torch.cuda.is_available() else "cpu"
            torch.manual_seed(seed)
            self.net = _Net(D, m, layers).to(self.dev)
            r = np.random.default_rng(seed)
            if cap_imgs and len(imgs) > cap_imgs:
                imgs = [imgs[i] for i in r.choice(len(imgs), cap_imgs, replace=False)]
            groups = {}                                          # 토큰수(N)별 배치
            for a in imgs:
                groups.setdefault(a.shape[0], []).append(l2(a))
            batches = [torch.tensor(np.stack(v), dtype=torch.float32) for v in groups.values()]
            opt = torch.optim.Adam(self.net.parameters(), lr=lr, weight_decay=1e-5)
            self.net.train()
            for ep in range(epochs):
                for b in batches:
                    xb = b.to(self.dev)
                    per = 1.0 - (self.net(xb, noise=noise) * xb).sum(-1)   # [B,N] 토큰별 코사인 손실
                    if hard_q < 1.0:                              # hard-mining: 어려운 토큰만
                        k = max(1, int(per.numel() * hard_q))
                        loss = torch.topk(per.flatten(), k).values.mean()
                    else:
                        loss = per.mean()
                    opt.zero_grad(); loss.backward(); opt.step()
            self.net.eval()
        def score(self, pf):
            x = torch.tensor(l2(pf)[None], dtype=torch.float32, device=self.dev)
            with torch.no_grad():
                r = self.net(x)                                  # eval: noise=0
                return (1.0 - (r * x).sum(-1))[0].cpu().numpy().astype(np.float32)
    return ReconLocalizer(normal_imgs)

if LOCALIZER == "recon":
    try:
        LOC = build_recon_localizer([feat[p][0] for p, ls in sup_items if not ls])
        print("localizer = recon (transformer, Dinomaly-lite)")
    except ImportError:
        LOC = KNNLocalizer(); print("torch 없음 → localizer = kNN 폴백")
else:
    LOC = KNNLocalizer(); print("localizer = kNN")

def anomaly(pf):
    return LOC.score(pf)

def typing_feats(pf, a, mode):
    idx = np.where(a > REGION_THR)[0]
    if len(idx) == 0:
        return np.zeros((0, pf.shape[1]), np.float32)
    P = l2(pf[idx])
    if mode == "residual":
        P = l2(P - nearest_normal(P))
    return P

# τ_region : 정상 support patch anomaly 분포 (선택된 localizer로 자기보정)
_ns = np.concatenate([anomaly(feat[p][0]) for p, ls in sup_items if not ls])
REGION_THR = float(np.percentile(_ns, REGION_PCT))
print("REGION_THR=%.4f  (정상 patch anomaly %g퍼센타일, localizer=%s)" % (REGION_THR, REGION_PCT, LOC.name))


## 3. 타입별 memory (단일라벨 support 이상 patch로 seed)


In [ ]:
def build_type_mem(mode):
    mems = {}
    for t in types:
        chunks = []
        for p, ls in sup_items:
            if ls == frozenset({t}):
                pf = feat[p][0]
                fc = typing_feats(pf, anomaly(pf), mode)
                if len(fc):
                    chunks.append(fc)
        if not chunks:
            print(f"  [경고] '{t}' 단일라벨 support 없음 → seed 불가(멀티라벨만 존재. MIL 필요)")
            mems[t] = None; continue
        allp = np.vstack(chunks)
        mems[t] = coreset(allp, MEM_SIZE)
        print(f"  {t}: 단일라벨 {len(chunks)}장 → 이상 patch {len(allp)} → memory {len(mems[t])}")
    return mems

MEMS = {m: build_type_mem(m) for m in ["raw", "residual"]}


## 4. 점수/예측 + 평가 (타입별 AP) — representation × head 분해

이전 비교(BG vs P-cosine)는 **전역/국소** 와 **logreg/cosine** 이 동시에 바뀌어 오염됨. 그래서 head 를
고정한 **P-logreg** 를 추가해 representation 효과만 격리한다.

| 이름 | representation | head |
|---|---|---|
| **BG** | 전역 평균 | OvR logreg |
| **P-log-raw** | 국소 이상 patch 평균(raw) | OvR logreg |
| **P-log-res** | 국소 이상 residual 평균 | OvR logreg |
| P-cos-raw/res | 국소(raw/res) | cosine memory |

- **BG vs P-log-res**: **오직 representation**(전역 vs 국소-맥락제거) 차이 → "국소에 신호 있나"의 정답.
- **P-log-res vs P-cos-res**: head(logreg vs cosine) 효과.
- AP = 타입별 threshold-free, multi-label 네이티브.


In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.linear_model import LogisticRegression

def type_scores(pf, mode, mems):
    a = anomaly(pf); F = typing_feats(pf, a, mode)
    sc = {t: 0.0 for t in types}
    if len(F) == 0:
        return sc, a
    for t in types:
        if mems[t] is None:
            continue
        col = (F @ mems[t].T).max(1)                     # 각 이상 patch의 c 최대매칭
        sc[t] = float(np.sort(col)[-TOPK:].mean())
    return sc, a

def eval_patch(mode):
    mems = MEMS[mode]
    Y = {t: [] for t in types}; S = {t: [] for t in types}
    for p, ls in qry_items:
        sc, _ = type_scores(feat[p][0], mode, mems)
        for t in types:
            Y[t].append(int(t in ls)); S[t].append(sc[t])
    return {t: (average_precision_score(Y[t], S[t]) if 0 < sum(Y[t]) < len(Y[t]) else float("nan"))
            for t in types}

def eval_baseline_global():
    # 이미지 전역평균 feature + 타입별 OvR logreg (support 전체로 학습)
    Xtr = np.vstack([l2(feat[p][0].mean(0)) for p, _ in sup_items])
    Xte = np.vstack([l2(feat[p][0].mean(0)) for p, _ in qry_items])
    ap = {}
    for t in types:
        ytr = np.array([int(t in ls) for _, ls in sup_items])
        yte = np.array([int(t in ls) for _, ls in qry_items])
        if len(set(ytr)) < 2 or 0 == sum(yte) or sum(yte) == len(yte):
            ap[t] = float("nan"); continue
        clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
        ap[t] = float(average_precision_score(yte, clf.predict_proba(Xte)[:, 1]))
    return ap

def _pooled_desc(pf, mode):
    # 국소 이상 patch(residual/raw) 평균 descriptor. BG 와 head 비교용.
    a = anomaly(pf); F = typing_feats(pf, a, mode)
    if len(F) == 0:                                  # 이상 patch 없으면 최이상 top-K 폴백(국소 유지)
        P = l2(pf[np.argsort(a)[-TOPK:]])
        F = l2(P - nearest_normal(P)) if mode == "residual" else P
    return l2(F.mean(0))

def eval_patch_logreg(mode):
    # 국소 descriptor + 타입별 OvR logreg (representation만 BG와 다름).
    Xtr = np.vstack([_pooled_desc(feat[p][0], mode) for p, _ in sup_items])
    Xte = np.vstack([_pooled_desc(feat[p][0], mode) for p, _ in qry_items])
    ap = {}
    for t in types:
        ytr = np.array([int(t in ls) for _, ls in sup_items])
        yte = np.array([int(t in ls) for _, ls in qry_items])
        if len(set(ytr)) < 2 or sum(yte) in (0, len(yte)):
            ap[t] = float("nan"); continue
        clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
        ap[t] = float(average_precision_score(yte, clf.predict_proba(Xte)[:, 1]))
    return ap

results = {"BG(전역)":   eval_baseline_global(),
           "P-log-raw":  eval_patch_logreg("raw"),
           "P-log-res":  eval_patch_logreg("residual"),
           "P-cos-raw":  eval_patch("raw"),
           "P-cos-res":  eval_patch("residual")}

try:
    import pandas as pd
    df = pd.DataFrame(results).T
    df["mAP"] = df.mean(axis=1)
    display(df.round(3))
except Exception:
    for k, v in results.items():
        print(k, {t: round(x, 3) for t, x in v.items()})

# viz/제품 기준 = 이긴 P-log-res: 타입별 logreg 저장(각 patch 에 적용 가능)
_Xtr = np.vstack([_pooled_desc(feat[p][0], "residual") for p, _ in sup_items])
HEADS_RES = {}
for t in types:
    ytr = np.array([int(t in ls) for _, ls in sup_items])
    HEADS_RES[t] = (LogisticRegression(max_iter=2000, class_weight="balanced").fit(_Xtr, ytr)
                    if len(set(ytr)) > 1 else None)


**읽는 법 (핵심 = BG vs P-log-res)**
- **P-log-res ≈/> BG** → 종류 신호가 **국소에 있음**. cosine 이 약했을 뿐 → typing 을 판별학습으로 전환. **방향 유효.**
- **P-log-res ≪ BG** → 전역이 진짜 더 가짐 → **shortcut 또는 결함성질**. 타입별로 갈라 봐야:
  - **국소 결함**(pitting=국소 뜯김/흠집): 여기서도 BG≫국소면 → shortcut 의심 강함.
  - **전역 결함**(partial=넓은 범위 부분 불일치): BG 우세가 **정당**(전역이 맞는 표현).
- **P-log-res vs P-cos-res**: head(logreg vs cosine) 효과. 크면 cosine 을 버리고 logreg 로.
- 타입별로 최적 전략이 다를 수 있음(국소→patch, 전역→global) 를 전제로 읽을 것.


## 5. others + 육안 — **이긴 모델(P-log-res)** per-patch 판정

각 이상 patch 의 residual 에 **타입별 logreg(P-log-res)** 를 적용 → argmax 타입으로 색칠.
모든 타입 확률이 낮으면(`< OTHERS_P`) **others**(미지). (cosine memory 아님 — 그건 최저성능이었음.)
위치가 실제 결함 위인지, 타입 배정이 맞는지 눈으로 확인.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

OTHERS_P = 0.5   # 모든 타입 logreg 확률 < 이 값이면 others(미지)

def _patch_proba(P):
    return np.stack([HEADS_RES[t].predict_proba(P)[:, 1] if HEADS_RES[t] is not None else np.zeros(len(P))
                     for t in types], axis=1)                   # [m, T] 타입별 확률

def show(n=8, others_p=OTHERS_P):
    picks = [it for it in qry_items if it[1]][:n]
    fig, ax = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
    ax = np.atleast_2d(ax)
    palette = plt.cm.tab10(np.linspace(0, 1, max(10, len(types))))
    for i, (p, ls) in enumerate(picks):
        pf, h, w = feat[p]; a = anomaly(pf)
        img = np.array(Image.open(p).convert("L").resize((192, 192)))
        ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title("|".join(sorted(ls))[:24]); ax[i, 0].axis("off")
        ax[i, 1].imshow(a.reshape(h, w), cmap="inferno"); ax[i, 1].set_title("anomaly"); ax[i, 1].axis("off")
        lab = -np.ones(h * w)                                   # -1=정상, -2=others, >=0 타입idx
        ridx = np.where(a > REGION_THR)[0]
        if len(ridx):
            P = l2(pf[ridx]); P = l2(P - nearest_normal(P))     # per-patch residual
            proba = _patch_proba(P); best = proba.argmax(1); bp = proba.max(1)
            lab[ridx] = np.where(bp >= others_p, best, -2)
        rgb = np.ones((h * w, 3)) * 0.15
        rgb[lab == -2] = [1, 1, 1]
        for ti in range(len(types)):
            rgb[lab == ti] = palette[ti][:3]
        ax[i, 2].imshow(rgb.reshape(h, w, 3)); ax[i, 2].axis("off")
        ax[i, 2].set_title("type/others (logreg)")
    plt.tight_layout(); plt.show()

show()
print("색=타입(tab10 순서: %s), 흰=others, 어두움=정상.  OTHERS_P=%.2f" % (", ".join(types), OTHERS_P))


## 6. 결론 메모

**확정(2026-07-23):** `P-log-res` 가 전 타입(저데이터 포함) 최고 > BG > `P-cos`.
→ 종류 신호는 **국소 + 맥락제거(residual) feature 에 실재**(shortcut 아님), **판별헤드(logreg)** 가 정답,
cosine memory 는 폐기. 다음: per-patch logreg(§5) 로 multi-label 출력 + others 임계 보정 → emclf_ui 재설계.

(참고) 실행 후 채울 것:
- BG vs P-raw vs P-residual **타입별 AP** →
- shortcut 확정?(전역이 낮고 patch 가 높음) →
- residual(맥락제거)이 raw 보다 나은가 →
- **localizer: knn vs recon** (EM_LOCALIZER=recon) — recon 위치정확이 AP로 이어지나 →
- IMG_SIZE 올리면 미세 타입 AP 오르나 →
- seed 부족 타입(단일라벨 없음)은? → 데이터 보강 or MIL 필요 →

**recon 개선 다음 단계**(효과順):
1. `RECON_HARDQ` 0.7~0.8 (어려운 정상 토큰 집중 → 잔차 선명)
2. `RECON_M`/`RECON_LAYERS`/`RECON_EPOCHS`↑ (용량·수렴)
3. **multi-layer 재구성**(여러 DINO layer 합침) = Dinomaly의 최대 레버, **서비스가 다층 patch 반환**해야 함 → 별도 작업
4. linear/unstable attention (identity-shortcut 추가 억제)

확정되면 `emclf_ui` 를 multi-label(타입별 memory+임계+others) 로 재설계.
